# Olympus Tech 4.0 Bonus Task: Rebuilding EDEN 

In [ ]:
# Imports and environment setup

import os
import re
import sys
import glob
import warnings
import subprocess
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

                                                                                
def install_packages():
    pkgs = ['lightgbm', 'xgboost', 'catboost', 'scikit-learn']
    for pkg in pkgs:
        try:
            __import__(pkg.replace('-', '_'))
        except ImportError:
            print(f"[Setup] Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_packages()

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import (
    HistGradientBoostingRegressor, HistGradientBoostingClassifier,
    ExtraTreesRegressor, ExtraTreesClassifier
)
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
)

                                                                                      
CONSOLE_LOGS = []
def log_print(*args, **kwargs):
    msg = " ".join(str(a) for a in args)
    CONSOLE_LOGS.append(msg)
    print(*args, **kwargs)


In [ ]:
# Dataset discovery and loading

def locate_datasets(search_dirs=('/content', '.')):
    found = {'train': None, 'test': None, 'sub': None}
    all_csvs = []
    for d in search_dirs:
        if os.path.exists(d):
            all_csvs.extend(glob.glob(os.path.join(d, '*.csv')))
    seen = set()
    unique_csvs = []
    for f in all_csvs:
        abs_f = os.path.abspath(f)
        if abs_f not in seen:
            seen.add(abs_f)
            unique_csvs.append(f)

    for f in unique_csvs:
        fname = os.path.basename(f).lower()
        if 'predicted' in fname:
            continue
        try:
            df_head = pd.read_csv(f, nrows=5)
            cols = set(df_head.columns)
            if 'settlement_index' in cols and len(cols) >= 15:
                if found['train'] is None or 'gentlemen' in fname:
                    found['train'] = f
            elif ('viable_planet' in cols or 'viable planet' in cols) and len(cols) <= 5:
                if found['sub'] is None or 'sample' in fname:
                    found['sub'] = f
            elif 'settlement_index' not in cols and len(cols) >= 15:
                if found['test'] is None or 'test' in fname:
                    found['test'] = f
        except Exception:
            continue

    if not all(found.values()):
        raise FileNotFoundError(
            f"Could not locate all required files. Found: {found}. "
            f"Available CSVs: {unique_csvs}"
        )
    return found['train'], found['test'], found['sub']

train_path, test_path, sub_path = locate_datasets()
log_print("DYNAMIC FILE DISCOVERY COMPLETE (V2 PIPELINE)")
log_print(f"  -> Cleaned Train Dataset : {train_path}")
log_print(f"  -> Unseen Test Dataset   : {test_path}")
log_print(f"  -> Sample Submission     : {sub_path}")

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
sub_raw = pd.read_csv(sub_path)

log_print(f"Train Shape: {train_raw.shape} | Test Shape: {test_raw.shape} | Sample Sub Shape: {sub_raw.shape}")


In [1]:
# Target preparation and leakage prevention

TARGET_REG_COL = 'settlement_index'
assert TARGET_REG_COL in train_raw.columns, "settlement_index must exist in training data!"

y_reg = train_raw[TARGET_REG_COL].values.astype(np.float64)
y_cls = (y_reg >= 60.0).astype(np.int32)

                                                                                        
X_train_raw = train_raw.drop(columns=[TARGET_REG_COL]).copy()
X_test_raw = test_raw.copy()

log_print(f"Target Distribution (viable_planet = settlement_index >= 60):")
log_print(f"  -> Viable (True / 1)     : {y_cls.sum()} ({y_cls.mean()*100:.2f}%)")
log_print(f"  -> Non-Viable (False / 0): {(1 - y_cls).sum()} ({(1 - y_cls.mean())*100:.2f}%)")


Target Distribution (viable_planet = settlement_index >= 60):
  -> Viable (True / 1)     : 2708 (42.31%)
  -> Non-Viable (False / 0): 3692 (57.69%)



In [1]:
# Data harmonization and feature engineering

GASES = ['N2', 'O2', 'Ar', 'CO2', 'CH4', 'H2O', 'He', 'H2', 'SO2', 'Ne', 'C2H6', 'NH3', 'H2S', 'CO']

def harmonize_star_type(val):
    if pd.isna(val):
        return 'M-type red dwarf'
    s = str(val).strip()
    sl = s.lower()
    if sl in ['red dwarf', 'red dwarf (m-type)', 'm-type main sequence', 'm-type red dwarf']:
        return 'M-type red dwarf'
    if sl in ['white dwarf', 'white dwarf remnant']:
        return 'White dwarf remnant'
    if 'k-type main sequence binary' in sl or 'k+k' in sl or 'red dwarf + k-type binary' in sl:
        return 'Binary system (K+K)'
    if ('f-type' in sl and 'g-type' in sl) or 'f+g' in sl:
        return 'Binary system (F+G)'
    if ('g-type' in sl and 'm-type' in sl) or 'g+m' in sl:
        return 'Binary system (G+M)'
    if 'subdwarf' in sl:
        return 'Subdwarf'
    return s

def harmonize_planet_type(val):
    if pd.isna(val):
        return 'terrestrial'
    s = str(val).strip().lower()
    if 'rogue' in s:
        return 'rogue planet'
    if 'relic' in s or 'artificial' in s:
        return 'relic/artificial world'
    if 'tidally locked' in s:
        return 'tidally locked'
    if 'super-earth' in s:
        return 'super-Earth'
    if 'gas giant moon' in s:
        return 'gas giant moon'
    if 'ocean' in s:
        return 'ocean world'
    if 'desert' in s:
        return 'desert world'
    if 'volcanic' in s:
        return 'volcanic world'
    if 'ice' in s:
        return 'ice world'
    if 'terrestrial' in s:
        return 'terrestrial'
    if 'crystalline' in s:
        return 'terrestrial'
    return s

def harmonize_water_presence(val):
    if pd.isna(val):
        return 'subsurface'
    s = str(val).strip().lower()
    if 'global ocean' in s:
        return 'global ocean'
    if 'seas' in s:
        return 'seas'
    if 'subsurface' in s:
        return 'subsurface'
    if 'trace' in s or 'ice' in s:
        return 'trace'
    if 'none' in s:
        return 'none'
    return 'subsurface'

def harmonize_surface_composition(val):
    if pd.isna(val):
        return 'rocky'
    s = str(val).strip().lower()
    first_tok = re.split(r'[+\s/]+', s)[0]
    mapping = {
        'rocky': 'rocky', 'rock': 'rocky', 'basaltic': 'volcanic', 'basalt': 'volcanic',
        'volcanic': 'volcanic', 'ice': 'ice', 'icy': 'ice', 'frozen': 'ice',
        'crystalline': 'crystalline', 'metallic': 'metallic', 'sandy': 'sandy'
    }
    return mapping.get(first_tok, 'rocky')

def harmonize_rings(val):
    if pd.isna(val):
        return 0
    s = str(val).strip().lower()
    return 1 if s in ['yes', 'true', '1'] else 0

def harmonize_primary_hazard(val):
    if pd.isna(val):
        return 'unknown_anomaly'
    s = str(val).strip().lower()
    if 'predator' in s or 'megafauna' in s:
        return 'predators'
    if 'toxic' in s:
        return 'toxic_air'
    if 'radiation' in s:
        return 'radiation'
    if 'storm' in s:
        return 'storms'
    if 'seismic' in s or 'volcanic' in s or 'thermal' in s:
        return 'seismic'
    if 'unknown' in s or 'anomaly' in s:
        return 'unknown_anomaly'
    return 'unknown_anomaly'

def build_features(df_in):
    df = df_in.copy()
    out = pd.DataFrame(index=df.index)

                                               
    out['name_prefix'] = df['planet_name'].astype(str).apply(lambda x: x.split('_')[0].strip())
    out['planet_id_num'] = df['planetId'].astype(float)

                                                                         
    for col in ['orbital_distance_au', 'orbital_period_days', 'day_length_hours',
                'axial_tilt_deg', 'escape_velocity_kms', 'oxygen_percent',
                'co2_percent', 'methane_percent']:
        out[f'isna_{col}'] = df[col].isna().astype(np.float32)
    out['total_missing_cols'] = out[[c for c in out.columns if c.startswith('isna_')]].sum(axis=1)

                                                                       
    raw_pt = df['planet_type'].fillna('').astype(str).str.lower()
    out['star_type'] = df['star_type'].apply(harmonize_star_type)
    out['planet_type'] = df['planet_type'].apply(harmonize_planet_type)
    out['is_tidally_locked_flag'] = raw_pt.str.contains('tidally locked').astype(np.float32)
    out['is_rogue_flag'] = (out['planet_type'] == 'rogue planet').astype(np.float32)
    out['is_binary_star'] = out['star_type'].str.contains('binary|Triple', case=False).astype(np.float32)

    raw_wp = df['water_presence'].fillna('').astype(str).str.lower()
    out['water_presence'] = df['water_presence'].apply(harmonize_water_presence)
    out['water_compound_flag'] = raw_wp.str.contains(r'\+|ice').astype(np.float32)

    raw_sc = df['surface_composition'].fillna('').astype(str).str.lower()
    out['surface_composition'] = df['surface_composition'].apply(harmonize_surface_composition)
    out['surface_compound_flag'] = raw_sc.str.contains(r'\+|\s').astype(np.float32)
    for surf in ['rocky', 'ice', 'volcanic', 'crystalline', 'metallic', 'sandy']:
        out[f'surf_has_{surf}'] = raw_sc.str.contains(surf[:4]).astype(np.float32)

    out['storm_frequency'] = df['storm_frequency'].fillna('seasonal').astype(str).str.strip().str.lower()
    out['seismic_activity'] = df['seismic_activity'].fillna('moderate').astype(str).str.strip().str.lower()
    out['magnetic_field'] = df['magnetic_field'].fillna('moderate').astype(str).str.strip().str.lower()
    out['radiation_level'] = df['radiation_level'].fillna('moderate').astype(str).str.strip().str.lower()
    out['rings_bin'] = df['rings'].apply(harmonize_rings).astype(np.float32)
    out['energy_harvest_potential'] = df['energy_harvest_potential'].fillna('moderate').astype(str).str.strip().str.lower()

    raw_hz = df['primary_hazard'].fillna('').astype(str).str.lower()
    out['primary_hazard'] = df['primary_hazard'].apply(harmonize_primary_hazard)
    out['hazard_temp_extreme_flag'] = raw_hz.str.contains('temp|cold|heat|thermal').astype(np.float32)
    out['hazard_compound_flag'] = raw_hz.str.contains(r'\+').astype(np.float32)

                                                          
    water_map = {'none': 0, 'trace': 1, 'subsurface': 2, 'seas': 3, 'global ocean': 4}
    storm_map = {'rare': 0, 'seasonal': 1, 'frequent': 2, 'constant': 3}
    level_map = {'low': 0, 'moderate': 1, 'high': 2, 'extreme': 3}
    mag_map = {'weak': 0, 'moderate': 1, 'strong': 2}

    out['water_ord'] = out['water_presence'].map(water_map).fillna(2).astype(np.float32)
    out['storm_ord'] = out['storm_frequency'].map(storm_map).fillna(1).astype(np.float32)
    out['seismic_ord'] = out['seismic_activity'].map(level_map).fillna(1).astype(np.float32)
    out['magnetic_ord'] = out['magnetic_field'].map(mag_map).fillna(1).astype(np.float32)
    out['radiation_ord'] = out['radiation_level'].map(level_map).fillna(1).astype(np.float32)
    out['energy_ord'] = out['energy_harvest_potential'].map(level_map).fillna(1).astype(np.float32)

                                                                    
    gas_Strings = df['atmosphere_primary_gases'].fillna('').astype(str)
    for g in GASES:
        pattern = rf'\b{g}\s*([0-9.]+)\s*%'
        extracted = gas_Strings.str.extract(pattern, expand=False).astype(float).fillna(0.0)
        out[f'parsed_{g}'] = extracted.values

    trace_pattern = r'(?:Trace|trace gases|trace|Tr)\s*([0-9.]+)\s*%'
    out['parsed_trace'] = gas_Strings.str.extract(trace_pattern, flags=re.IGNORECASE, expand=False).astype(float).fillna(0.0).values
    out['parsed_num_gases'] = (out[[f'parsed_{g}' for g in GASES]] > 0).sum(axis=1).astype(np.float32)

    def clean_atmo_str(s):
        s = re.sub(r'[,+]', ' ', str(s))
        s = re.sub(r'trace gases|trace|Tr', 'Trace', s, flags=re.IGNORECASE)
        s = re.sub(r'\s+', ' ', s).strip()
        return s
    out['atmo_canonical'] = gas_Strings.apply(clean_atmo_str)

                                                                         
    o2_raw = df['oxygen_percent'].copy()
    o2_fill = np.where(out['parsed_O2'] > 0, out['parsed_O2'], 0.05)
    out['oxygen_percent'] = np.where(o2_raw.isna(), o2_fill, o2_raw)
    out['o2_diff_parsed'] = out['oxygen_percent'] - out['parsed_O2']

    co2_raw = df['co2_percent'].copy()
    co2_fill = np.where(out['parsed_CO2'] > 0, out['parsed_CO2'], 0.15)
    out['co2_percent'] = np.where(co2_raw.isna(), co2_fill, co2_raw)
    out['co2_diff_parsed'] = out['co2_percent'] - out['parsed_CO2']

    ch4_raw = df['methane_percent'].copy()
    ch4_fill = np.where(out['parsed_CH4'] > 0, out['parsed_CH4'], 0.05)
    out['methane_percent'] = np.where(ch4_raw.isna(), ch4_fill, ch4_raw)
    out['ch4_diff_parsed'] = out['methane_percent'] - out['parsed_CH4']

                                                          
    orb_dist = df['orbital_distance_au'].copy()
    orb_per = df['orbital_period_days'].copy()
    orb_dist = np.where(orb_dist.isna() & (out['planet_type'] == 'rogue planet'), 0.0, orb_dist)
    orb_per  = np.where(orb_per.isna()  & (out['planet_type'] == 'rogue planet'), 0.0, orb_per)
    out['orbital_distance_au'] = orb_dist
    out['orbital_period_days'] = orb_per

    out['day_length_hours'] = df['day_length_hours'].values
    out['axial_tilt_deg'] = df['axial_tilt_deg'].values
    out['mean_temp_c'] = df['mean_temp_c'].values
    out['temp_range_c'] = df['temp_range_c'].values
    out['gravity_g'] = df['gravity_g'].values
    out['escape_velocity_kms'] = df['escape_velocity_kms'].values
    out['atmospheric_pressure_atm'] = df['atmospheric_pressure_atm'].values
    out['albedo'] = df['albedo'].values
    out['cloud_coverage_percent'] = df['cloud_coverage_percent'].values
    out['moons_count'] = df['moons_count'].fillna(2.0).values
    out['rare_mineral_index'] = df['rare_mineral_index'].values

                                                                                           
    out['cloud_is_fractional'] = (out['cloud_coverage_percent'] <= 1.0).astype(np.float32)
    out['cloud_unified_pct'] = np.where(
        out['cloud_coverage_percent'] <= 1.0,
        out['cloud_coverage_percent'] * 100.0,
        out['cloud_coverage_percent']
    )

    out['axial_tilt_gt1'] = (out['axial_tilt_deg'] > 1.0).astype(np.float32)
    out['axial_tilt_unified'] = np.where(
        out['axial_tilt_deg'] <= 1.0,
        out['axial_tilt_deg'] * 23.5,
        out['axial_tilt_deg']
    )

                                                                                         
    out['orbital_period_is_logged'] = ((out['orbital_period_days'] > 0.0) & (out['orbital_period_days'] <= 10.0)).astype(np.float32)
    out['orbital_period_unified_days'] = np.where(
        (out['orbital_period_days'] > 0.0) & (out['orbital_period_days'] <= 10.0),
        np.exp(np.clip(out['orbital_period_days'], 0.0, 10.0)),
        out['orbital_period_days']
    )
    out['orbital_period_unified_log'] = np.where(
        out['orbital_period_days'] > 10.0,
        np.log(np.clip(out['orbital_period_days'], 1.0, None)),
        out['orbital_period_days']
    )

    for col in ['orbital_distance_au', 'orbital_period_days', 'day_length_hours',
                'axial_tilt_deg', 'mean_temp_c', 'temp_range_c', 'gravity_g',
                'escape_velocity_kms', 'atmospheric_pressure_atm', 'albedo', 'cloud_coverage_percent']:
        s = out[col].fillna(0.0)
        out[f'{col}_is_round2'] = (np.abs(s - np.round(s, 2)) < 1e-5).astype(np.float32)

    out['total_round2_cols'] = out[[c for c in out.columns if c.endswith('_is_round2')]].sum(axis=1)

                                                      
    out['p_O2_atm'] = out['atmospheric_pressure_atm'] * (out['oxygen_percent'] / 100.0)
    out['p_CO2_atm'] = out['atmospheric_pressure_atm'] * (out['co2_percent'] / 100.0)
    out['p_CH4_atm'] = out['atmospheric_pressure_atm'] * (out['methane_percent'] / 100.0)
    out['p_N2_atm'] = out['atmospheric_pressure_atm'] * (out['parsed_N2'] / 100.0)
    out['toxic_gas_sum'] = out['co2_percent'] + out['methane_percent'] + out['parsed_SO2'] + out['parsed_H2S'] + out['parsed_NH3'] + out['parsed_CO']
    out['inert_gas_sum'] = out['parsed_N2'] + out['parsed_Ar'] + out['parsed_He'] + out['parsed_Ne']
    out['breathability_ratio'] = out['oxygen_percent'] / (out['toxic_gas_sum'] + 1.0)

    out['temp_dev_earth'] = np.abs(out['mean_temp_c'] - 15.0)
    out['temp_in_liquid_water_band'] = ((out['mean_temp_c'] >= -10.0) & (out['mean_temp_c'] <= 45.0)).astype(np.float32)
    out['min_temp_c'] = out['mean_temp_c'] - 0.5 * out['temp_range_c']
    out['max_temp_c'] = out['mean_temp_c'] + 0.5 * out['temp_range_c']
    out['gravity_dev_earth'] = np.abs(out['gravity_g'] - 1.0)
    out['gravity_comfortable'] = ((out['gravity_g'] >= 0.5) & (out['gravity_g'] <= 1.3)).astype(np.float32)
    out['pressure_dev_earth'] = np.abs(out['atmospheric_pressure_atm'] - 1.0)
    out['pressure_comfortable'] = ((out['atmospheric_pressure_atm'] >= 0.4) & (out['atmospheric_pressure_atm'] <= 2.2)).astype(np.float32)
    out['o2_dev_earth'] = np.abs(out['oxygen_percent'] - 21.0)
    out['o2_breathable_band'] = ((out['oxygen_percent'] >= 12.0) & (out['oxygen_percent'] <= 26.0)).astype(np.float32)

    ve_norm = out['escape_velocity_kms'] / 11.186
    out['radius_proxy_earth'] = (ve_norm ** 2) / np.clip(out['gravity_g'], 0.02, None)
    out['mass_proxy_earth'] = (ve_norm ** 4) / np.clip(out['gravity_g'], 0.02, None)
    out['ve_to_gravity_ratio'] = out['escape_velocity_kms'] / np.clip(out['gravity_g'], 0.02, None)

                                                  
    period_years_unified = out['orbital_period_unified_days'] / 365.25
    out['stellar_mass_proxy'] = np.where(
        out['orbital_distance_au'] > 0.01,
        (out['orbital_distance_au'] ** 3) / np.clip(period_years_unified ** 2, 1e-6, None),
        0.0
    )
    out['kepler_expected_days'] = 365.25 * (np.clip(out['orbital_distance_au'], 0.0, None) ** 1.5)
    out['kepler_period_ratio'] = out['orbital_period_unified_days'] / (out['kepler_expected_days'] + 1.0)

    out['log_orbital_period'] = np.log1p(np.clip(out['orbital_period_days'], 0.0, None))
    out['log_orbital_distance'] = np.log1p(np.clip(out['orbital_distance_au'], 0.0, None))
    out['log_day_length'] = np.log1p(np.clip(out['day_length_hours'], 0.0, None))
    out['log_temp_range'] = np.log1p(np.clip(out['temp_range_c'], 0.0, None))

    out['shielding_score'] = out['magnetic_ord'] * 2.0 - out['radiation_ord'] - out['storm_ord'] - out['seismic_ord']
    out['resource_score'] = (out['rare_mineral_index'] / 100.0) + 0.25 * out['energy_ord'] + 0.25 * out['water_ord']
    out['eden_heuristic_index'] = (
        0.35 * np.clip(out['oxygen_percent'] / 21.0, 0, 1.3)
        + 0.20 * np.exp(-out['temp_dev_earth'] / 50.0)
        + 0.15 * np.exp(-out['gravity_dev_earth'] / 0.5)
        + 0.15 * (out['water_ord'] / 4.0)
        + 0.15 * ((out['shielding_score'] + 9.0) / 13.0)
    )

                                                 
    out['o2_x_gravity'] = out['oxygen_percent'] * out['gravity_g']
    out['o2_x_axial_tilt'] = out['oxygen_percent'] * out['axial_tilt_deg']
    out['gravity_x_axial_tilt'] = out['gravity_g'] * out['axial_tilt_deg']
    out['o2_x_water_ord'] = out['oxygen_percent'] * (out['water_ord'] + 1.0)
    out['effective_solar_flux'] = (1.0 - out['albedo']) / np.clip(out['orbital_distance_au'] ** 2 + 0.05, 0.05, None)

    return out


[Step 3] Engineering V2 domain, chemical, and unified physical features...


In [1]:
# Feature generation and preprocessing

log_print("\n[Step 3] Engineering V2 domain, chemical, and unified physical features...")
df_all_raw = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
df_all_feat = build_features(df_all_raw)

for col in ['orbital_distance_au', 'orbital_period_days', 'day_length_hours', 'axial_tilt_deg', 'escape_velocity_kms']:
    df_all_feat[col] = df_all_feat.groupby('planet_type')[col].transform(lambda s: s.fillna(s.median()))
    df_all_feat[col] = df_all_feat[col].fillna(df_all_feat[col].median())

                                                            
ve_norm = df_all_feat['escape_velocity_kms'] / 11.186
df_all_feat['radius_proxy_earth'] = (ve_norm ** 2) / np.clip(df_all_feat['gravity_g'], 0.02, None)
df_all_feat['mass_proxy_earth'] = (ve_norm ** 4) / np.clip(df_all_feat['gravity_g'], 0.02, None)
df_all_feat['ve_to_gravity_ratio'] = df_all_feat['escape_velocity_kms'] / np.clip(df_all_feat['gravity_g'], 0.02, None)
df_all_feat['axial_tilt_gt1'] = (df_all_feat['axial_tilt_deg'] > 1.0).astype(np.float32)
df_all_feat['axial_tilt_unified'] = np.where(
    df_all_feat['axial_tilt_deg'] <= 1.0,
    df_all_feat['axial_tilt_deg'] * 23.5,
    df_all_feat['axial_tilt_deg']
)
df_all_feat['orbital_period_is_logged'] = ((df_all_feat['orbital_period_days'] > 0.0) & (df_all_feat['orbital_period_days'] <= 10.0)).astype(np.float32)
df_all_feat['orbital_period_unified_days'] = np.where(
    (df_all_feat['orbital_period_days'] > 0.0) & (df_all_feat['orbital_period_days'] <= 10.0),
    np.exp(np.clip(df_all_feat['orbital_period_days'], 0.0, 10.0)),
    df_all_feat['orbital_period_days']
)
df_all_feat['orbital_period_unified_log'] = np.where(
    df_all_feat['orbital_period_days'] > 10.0,
    np.log(np.clip(df_all_feat['orbital_period_days'], 1.0, None)),
    df_all_feat['orbital_period_days']
)
df_all_feat['log_orbital_period'] = np.log1p(np.clip(df_all_feat['orbital_period_days'], 0.0, None))
df_all_feat['log_orbital_distance'] = np.log1p(np.clip(df_all_feat['orbital_distance_au'], 0.0, None))
df_all_feat['log_day_length'] = np.log1p(np.clip(df_all_feat['day_length_hours'], 0.0, None))
df_all_feat['o2_x_axial_tilt'] = df_all_feat['oxygen_percent'] * df_all_feat['axial_tilt_deg']
df_all_feat['gravity_x_axial_tilt'] = df_all_feat['gravity_g'] * df_all_feat['axial_tilt_deg']

                                     
df_all_feat['star_planet_pair'] = df_all_feat['star_type'].astype(str) + "__" + df_all_feat['planet_type'].astype(str)
df_all_feat['water_surface_pair'] = df_all_feat['water_presence'].astype(str) + "__" + df_all_feat['surface_composition'].astype(str)

CAT_COLS = [
    'name_prefix', 'star_type', 'planet_type', 'water_presence',
    'surface_composition', 'storm_frequency', 'seismic_activity',
    'magnetic_field', 'radiation_level', 'energy_harvest_potential',
    'primary_hazard', 'atmo_canonical', 'star_planet_pair', 'water_surface_pair'
]

for c in CAT_COLS:
    freq = df_all_feat[c].value_counts(normalize=True)
    df_all_feat[f'{c}_freq'] = df_all_feat[c].map(freq).astype(np.float32)

for grp in ['planet_type', 'star_type', 'water_presence']:
    for num_c in ['oxygen_percent', 'gravity_g', 'mean_temp_c', 'axial_tilt_deg', 'cloud_coverage_percent']:
        grp_mean = df_all_feat.groupby(grp)[num_c].transform('mean')
        grp_std = df_all_feat.groupby(grp)[num_c].transform('std').replace(0, 1.0)
        df_all_feat[f'{num_c}_diff_{grp}'] = df_all_feat[num_c] - grp_mean
        df_all_feat[f'{num_c}_z_{grp}'] = (df_all_feat[num_c] - grp_mean) / grp_std

n_train = len(X_train_raw)
X_train = df_all_feat.iloc[:n_train].copy().reset_index(drop=True)
X_test = df_all_feat.iloc[n_train:].copy().reset_index(drop=True)

X_train_mod = X_train.drop(columns=['planet_id_num']).copy()
X_test_mod = X_test.drop(columns=['planet_id_num']).copy()

log_print(f"[Step 3] V2 Feature Engineering Complete! Total Engineered Features: {X_train_mod.shape[1]}")


[Step 3] V2 Feature Engineering Complete! Total Engineered Features: 176



In [ ]:
# Ensemble configuration

N_SPLITS = 10
RANDOM_SEED = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

model_names = [
    'lgb_reg', 'lgb_cls',
    'cat_reg', 'cat_cls',
    'xgb_reg', 'xgb_cls',
    'hgb_reg', 'hgb_cls',
    'et_reg',  'et_cls',
    'mlp_reg', 'mlp_cls'
]
oof_preds = {m: np.zeros(n_train, dtype=np.float64) for m in model_names}
test_preds = {m: np.zeros(len(X_test_mod), dtype=np.float64) for m in model_names}

log_print(f"\n[Step 4] Training {N_SPLITS}-Fold 12-Model Dual-Objective Ensemble (GBDTs + ExtraTrees + Neural MLP)...")


In [1]:
# Cross-validation and base-model training

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_mod, y_cls), 1):
    X_tr = X_train_mod.iloc[tr_idx].copy()
    X_va = X_train_mod.iloc[va_idx].copy()
    X_te = X_test_mod.copy()

    y_tr_reg, y_va_reg = y_reg[tr_idx], y_reg[va_idx]
    y_tr_cls, y_va_cls = y_cls[tr_idx], y_cls[va_idx]

                                                                        
    global_mean_reg = y_tr_reg.mean()
    global_mean_cls = y_tr_cls.mean()
    smoothing = 15.0

    for c in CAT_COLS:
        stats_reg = pd.Series(y_tr_reg).groupby(X_tr[c].values).agg(['count', 'mean'])
        smooth_reg = (stats_reg['count'] * stats_reg['mean'] + smoothing * global_mean_reg) / (stats_reg['count'] + smoothing)

        stats_cls = pd.Series(y_tr_cls).groupby(X_tr[c].values).agg(['count', 'mean'])
        smooth_cls = (stats_cls['count'] * stats_cls['mean'] + smoothing * global_mean_cls) / (stats_cls['count'] + smoothing)

        X_tr[f'{c}_te_reg'] = X_tr[c].map(smooth_reg).fillna(global_mean_reg).astype(np.float32)
        X_va[f'{c}_te_reg'] = X_va[c].map(smooth_reg).fillna(global_mean_reg).astype(np.float32)
        X_te[f'{c}_te_reg'] = X_te[c].map(smooth_reg).fillna(global_mean_reg).astype(np.float32)

        X_tr[f'{c}_te_cls'] = X_tr[c].map(smooth_cls).fillna(global_mean_cls).astype(np.float32)
        X_va[f'{c}_te_cls'] = X_va[c].map(smooth_cls).fillna(global_mean_cls).astype(np.float32)
        X_te[f'{c}_te_cls'] = X_te[c].map(smooth_cls).fillna(global_mean_cls).astype(np.float32)

    for c in CAT_COLS:
        vocab = sorted(list(set(X_tr[c].astype(str)).union(set(X_va[c].astype(str))).union(set(X_te[c].astype(str)))))
        cat_type = pd.CategoricalDtype(categories=vocab, ordered=False)
        X_tr[c] = X_tr[c].astype(str).astype(cat_type)
        X_va[c] = X_va[c].astype(str).astype(cat_type)
        X_te[c] = X_te[c].astype(str).astype(cat_type)

    num_feature_cols = [c for c in X_tr.columns if c not in CAT_COLS]
    X_tr_num = X_tr[num_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_va_num = X_va[num_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_te_num = X_te[num_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

                                                    
    model_lgb_reg = lgb.LGBMRegressor(
        n_estimators=2200, learning_rate=0.02, num_leaves=31, max_depth=6,
        min_child_samples=25, subsample=0.8, subsample_freq=1, colsample_bytree=0.72,
        reg_alpha=1.2, reg_lambda=2.5, random_state=RANDOM_SEED + fold, n_jobs=-1, verbose=-1
    )
    model_lgb_reg.fit(X_tr, y_tr_reg, eval_set=[(X_va, y_va_reg)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_preds['lgb_reg'][va_idx] = model_lgb_reg.predict(X_va)
    test_preds['lgb_reg'] += model_lgb_reg.predict(X_te) / N_SPLITS

    model_lgb_cls = lgb.LGBMClassifier(
        n_estimators=2200, learning_rate=0.02, num_leaves=31, max_depth=6,
        min_child_samples=25, subsample=0.8, subsample_freq=1, colsample_bytree=0.72,
        reg_alpha=1.2, reg_lambda=2.5, random_state=RANDOM_SEED + 100 + fold, n_jobs=-1, verbose=-1
    )
    model_lgb_cls.fit(X_tr, y_tr_cls, eval_set=[(X_va, y_va_cls)], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_preds['lgb_cls'][va_idx] = model_lgb_cls.predict_proba(X_va)[:, 1]
    test_preds['lgb_cls'] += model_lgb_cls.predict_proba(X_te)[:, 1] / N_SPLITS

                                                    
    X_tr_cb, X_va_cb, X_te_cb = X_tr.copy(), X_va.copy(), X_te.copy()
    for c in CAT_COLS:
        X_tr_cb[c] = X_tr_cb[c].astype(str)
        X_va_cb[c] = X_va_cb[c].astype(str)
        X_te_cb[c] = X_te_cb[c].astype(str)

    model_cat_reg = CatBoostRegressor(
        iterations=2500, learning_rate=0.03, depth=6, l2_leaf_reg=4.5,
        cat_features=CAT_COLS, random_seed=RANDOM_SEED + 200 + fold, verbose=0
    )
    model_cat_reg.fit(X_tr_cb, y_tr_reg, eval_set=(X_va_cb, y_va_reg), early_stopping_rounds=100, verbose=False)
    oof_preds['cat_reg'][va_idx] = model_cat_reg.predict(X_va_cb)
    test_preds['cat_reg'] += model_cat_reg.predict(X_te_cb) / N_SPLITS

    model_cat_cls = CatBoostClassifier(
        iterations=2500, learning_rate=0.03, depth=6, l2_leaf_reg=4.5,
        cat_features=CAT_COLS, random_seed=RANDOM_SEED + 300 + fold, verbose=0
    )
    model_cat_cls.fit(X_tr_cb, y_tr_cls, eval_set=(X_va_cb, y_va_cls), early_stopping_rounds=100, verbose=False)
    oof_preds['cat_cls'][va_idx] = model_cat_cls.predict_proba(X_va_cb)[:, 1]
    test_preds['cat_cls'] += model_cat_cls.predict_proba(X_te_cb)[:, 1] / N_SPLITS

                                                   
    model_xgb_reg = xgb.XGBRegressor(
        n_estimators=2200, learning_rate=0.02, max_depth=5, subsample=0.8,
        colsample_bytree=0.72, reg_alpha=1.2, reg_lambda=3.0, min_child_weight=22,
        enable_categorical=True, tree_method='hist', early_stopping_rounds=100,
        random_state=RANDOM_SEED + 400 + fold, n_jobs=-1, verbosity=0
    )
    model_xgb_reg.fit(X_tr, y_tr_reg, eval_set=[(X_va, y_va_reg)], verbose=False)
    oof_preds['xgb_reg'][va_idx] = model_xgb_reg.predict(X_va)
    test_preds['xgb_reg'] += model_xgb_reg.predict(X_te) / N_SPLITS

    model_xgb_cls = xgb.XGBClassifier(
        n_estimators=2200, learning_rate=0.02, max_depth=5, subsample=0.8,
        colsample_bytree=0.72, reg_alpha=1.2, reg_lambda=3.0, min_child_weight=22,
        enable_categorical=True, tree_method='hist', early_stopping_rounds=100,
        eval_metric='logloss', random_state=RANDOM_SEED + 500 + fold, n_jobs=-1, verbosity=0
    )
    model_xgb_cls.fit(X_tr, y_tr_cls, eval_set=[(X_va, y_va_cls)], verbose=False)
    oof_preds['xgb_cls'][va_idx] = model_xgb_cls.predict_proba(X_va)[:, 1]
    test_preds['xgb_cls'] += model_xgb_cls.predict_proba(X_te)[:, 1] / N_SPLITS

                                                                
    model_hgb_reg = HistGradientBoostingRegressor(
        max_iter=700, learning_rate=0.025, max_leaf_nodes=31, max_depth=6,
        min_samples_leaf=25, l2_regularization=2.5, early_stopping=True,
        random_state=RANDOM_SEED + 600 + fold
    )
    model_hgb_reg.fit(X_tr_num, y_tr_reg)
    oof_preds['hgb_reg'][va_idx] = model_hgb_reg.predict(X_va_num)
    test_preds['hgb_reg'] += model_hgb_reg.predict(X_te_num) / N_SPLITS

    model_hgb_cls = HistGradientBoostingClassifier(
        max_iter=700, learning_rate=0.025, max_leaf_nodes=31, max_depth=6,
        min_samples_leaf=25, l2_regularization=2.5, early_stopping=True,
        random_state=RANDOM_SEED + 700 + fold
    )
    model_hgb_cls.fit(X_tr_num, y_tr_cls)
    oof_preds['hgb_cls'][va_idx] = model_hgb_cls.predict_proba(X_va_num)[:, 1]
    test_preds['hgb_cls'] += model_hgb_cls.predict_proba(X_te_num)[:, 1] / N_SPLITS

                                                                                    
    model_et_reg = ExtraTreesRegressor(
        n_estimators=300, max_depth=16, min_samples_leaf=6, max_features=0.65,
        random_state=RANDOM_SEED + 800 + fold, n_jobs=-1
    )
    model_et_reg.fit(X_tr_num, y_tr_reg)
    oof_preds['et_reg'][va_idx] = model_et_reg.predict(X_va_num)
    test_preds['et_reg'] += model_et_reg.predict(X_te_num) / N_SPLITS

    model_et_cls = ExtraTreesClassifier(
        n_estimators=300, max_depth=16, min_samples_leaf=6, max_features=0.65,
        random_state=RANDOM_SEED + 900 + fold, n_jobs=-1
    )
    model_et_cls.fit(X_tr_num, y_tr_cls)
    oof_preds['et_cls'][va_idx] = model_et_cls.predict_proba(X_va_num)[:, 1]
    test_preds['et_cls'] += model_et_cls.predict_proba(X_te_num)[:, 1] / N_SPLITS

                                                                                            
    scaler = RobustScaler()
    X_tr_scaled = np.clip(scaler.fit_transform(X_tr_num), -8.0, 8.0)
    X_va_scaled = np.clip(scaler.transform(X_va_num), -8.0, 8.0)
    X_te_scaled = np.clip(scaler.transform(X_te_num), -8.0, 8.0)

    model_mlp_reg = MLPRegressor(
        hidden_layer_sizes=(128, 64), activation='relu', alpha=2.0,
        learning_rate_init=0.001, max_iter=200, early_stopping=True,
        random_state=RANDOM_SEED + 1000 + fold
    )
    model_mlp_reg.fit(X_tr_scaled, y_tr_reg)
    oof_preds['mlp_reg'][va_idx] = model_mlp_reg.predict(X_va_scaled)
    test_preds['mlp_reg'] += model_mlp_reg.predict(X_te_scaled) / N_SPLITS

    model_mlp_cls = MLPClassifier(
        hidden_layer_sizes=(128, 64), activation='relu', alpha=2.0,
        learning_rate_init=0.001, max_iter=200, early_stopping=True,
        random_state=RANDOM_SEED + 1100 + fold
    )
    model_mlp_cls.fit(X_tr_scaled, y_tr_cls)
    oof_preds['mlp_cls'][va_idx] = model_mlp_cls.predict_proba(X_va_scaled)[:, 1]
    test_preds['mlp_cls'] += model_mlp_cls.predict_proba(X_te_scaled)[:, 1] / N_SPLITS

    fold_blend_acc = accuracy_score(
        y_va_cls,
        (0.5 * expit((oof_preds['cat_reg'][va_idx] - 60.0) / 2.0) + 0.5 * oof_preds['cat_cls'][va_idx] >= 0.5).astype(int)
    )
    log_print(f"  Fold {fold:02d}/{N_SPLITS} | CatBoost Reg RMSE: {np.sqrt(mean_squared_error(y_va_reg, oof_preds['cat_reg'][va_idx])):.4f} | Fold Acc: {fold_blend_acc*100:.2f}%")


[Step 4] Training 10-Fold 12-Model Dual-Objective Ensemble (GBDTs + ExtraTrees + Neural MLP)...
  Fold 01/10 | CatBoost Reg RMSE: 4.5710 | Fold Acc: 87.03%
  Fold 02/10 | CatBoost Reg RMSE: 4.3124 | Fold Acc: 88.91%
  Fold 03/10 | CatBoost Reg RMSE: 4.2312 | Fold Acc: 90.16%
  Fold 04/10 | CatBoost Reg RMSE: 4.3190 | Fold Acc: 87.19%
  Fold 05/10 | CatBoost Reg RMSE: 3.8064 | Fold Acc: 89.53%
  Fold 06/10 | CatBoost Reg RMSE: 4.3347 | Fold Acc: 90.16%
  Fold 07/10 | CatBoost Reg RMSE: 4.5784 | Fold Acc: 87.50%
  Fold 08/10 | CatBoost Reg RMSE: 4.2517 | Fold Acc: 90.31%
  Fold 08/10 | CatBoost Reg RMSE: 4.2517 | Fold Acc: 90.31%
  Fold 09/10 | CatBoost Reg RMSE: 4.4564 | Fold Acc: 86.09%
  Fold 09/10 | CatBoost Reg RMSE: 4.4564 | Fold Acc: 86.09%
  Fold 10/10 | CatBoost Reg RMSE: 4.6110 | Fold Acc: 88.12%




In [ ]:
# Base-model performance evaluation

log_print("OUT-OF-FOLD PERFORMANCE SUMMARY (12 BASE MODELS)")
for m in model_names:
    if m.endswith('_reg'):
        rmse = np.sqrt(mean_squared_error(y_reg, oof_preds[m]))
        r2 = r2_score(y_reg, oof_preds[m])
        acc = accuracy_score(y_cls, (oof_preds[m] >= 60.0).astype(int))
        auc = roc_auc_score(y_cls, oof_preds[m])
        log_print(f"  {m:10s} | RMSE: {rmse:.4f} | R2: {r2:.4f} | Acc (@60.0): {acc*100:.3f}% | ROC-AUC: {auc:.5f}")
    else:
        acc = accuracy_score(y_cls, (oof_preds[m] >= 0.5).astype(int))
        f1 = f1_score(y_cls, (oof_preds[m] >= 0.5).astype(int))
        auc = roc_auc_score(y_cls, oof_preds[m])
        log_print(f"  {m:10s} | Acc (@0.50): {acc*100:.3f}% | F1: {f1:.4f} | ROC-AUC: {auc:.5f}")


OUT-OF-FOLD PERFORMANCE SUMMARY (12 BASE MODELS)

  lgb_reg    | RMSE: 4.5238 | R2: 0.8238 | Acc (@60.0): 87.812% | ROC-AUC: 0.94277
  lgb_cls    | Acc (@0.50): 87.484% | F1: 0.8466 | ROC-AUC: 0.93898
  cat_reg    | RMSE: 4.3530 | R2: 0.8369 | Acc (@60.0): 88.625% | ROC-AUC: 0.94739
  cat_cls    | Acc (@0.50): 88.141% | F1: 0.8551 | ROC-AUC: 0.94474
  xgb_reg    | RMSE: 4.8892 | R2: 0.7942 | Acc (@60.0): 87.016% | ROC-AUC: 0.93676
  xgb_cls    | Acc (@0.50): 87.703% | F1: 0.8501 | ROC-AUC: 0.94050
  hgb_reg    | RMSE: 4.4998 | R2: 0.8257 | Acc (@60.0): 87.812% | ROC-AUC: 0.94375
  hgb_cls    | Acc (@0.50): 87.266% | F1: 0.8447 | ROC-AUC: 0.93792
  et_reg     | RMSE: 4.8142 | R2: 0.8005 | Acc (@60.0): 86.500% | ROC-AUC: 0.93378
  et_cls     | Acc (@0.50): 86.141% | F1: 0.8287 | ROC-AUC: 0.92897
  mlp_reg    | RMSE: 5.2004 | R2: 0.7672 | Acc (@60.0): 85.219% | ROC-AUC: 0.92637
  mlp_cls    | Acc (@0.50): 86.047% | F1: 0.8303 | ROC-AUC: 0.92845
  Fold 10/10 | CatBoost Reg RMSE: 4.6110 | F

In [1]:
# Stacking and meta-model training

reg_cols = ['cat_reg', 'lgb_reg', 'xgb_reg', 'hgb_reg', 'et_reg', 'mlp_reg']
cls_cols = ['cat_cls', 'lgb_cls', 'xgb_cls', 'hgb_cls', 'et_cls', 'mlp_cls']

S_oof_reg = np.column_stack([oof_preds[c] for c in reg_cols])
S_test_reg = np.column_stack([test_preds[c] for c in reg_cols])

S_oof_cls = np.column_stack([oof_preds[c] for c in cls_cols])
S_test_cls = np.column_stack([test_preds[c] for c in cls_cols])

oof_reg_blend = np.zeros(n_train)
test_reg_blend = np.zeros(len(X_test_mod))
oof_cls_blend = np.zeros(n_train)
test_cls_blend = np.zeros(len(X_test_mod))

                                                        
for tr_idx, va_idx in skf.split(X_train_mod, y_cls):
    meta_reg = RidgeCV(alphas=np.logspace(-1, 4, 30))
    meta_reg.fit(S_oof_reg[tr_idx], y_reg[tr_idx])
    oof_reg_blend[va_idx] = meta_reg.predict(S_oof_reg[va_idx])
    test_reg_blend += meta_reg.predict(S_test_reg) / N_SPLITS

                                                                                           
    X_meta_tr = np.column_stack([expit((S_oof_reg[tr_idx] - 60.0) / 2.2), S_oof_cls[tr_idx]])
    X_meta_va = np.column_stack([expit((S_oof_reg[va_idx] - 60.0) / 2.2), S_oof_cls[va_idx]])
    X_meta_te = np.column_stack([expit((S_test_reg - 60.0) / 2.2), S_test_cls])

    meta_cls = LogisticRegressionCV(Cs=20, cv=5, scoring='neg_log_loss', max_iter=500)
    meta_cls.fit(X_meta_tr, y_cls[tr_idx])
    oof_cls_blend[va_idx] = meta_cls.predict_proba(X_meta_va)[:, 1]
    test_cls_blend += meta_cls.predict_proba(X_meta_te)[:, 1] / N_SPLITS

log_print(f"\n  -> Meta-Regressor OOF RMSE : {np.sqrt(mean_squared_error(y_reg, oof_reg_blend)):.4f} | R2: {r2_score(y_reg, oof_reg_blend):.4f}")
log_print(f"  -> Meta-Classifier OOF AUC : {roc_auc_score(y_cls, oof_cls_blend):.5f}")


  -> Meta-Regressor OOF RMSE : 4.3533 | R2: 0.8368
  -> Meta-Classifier OOF AUC : 0.94719

  -> Meta-Regressor OOF RMSE : 4.3533 | R2: 0.8368
  -> Meta-Classifier OOF AUC : 0.94719




In [1]:
# Blend optimization and final evaluation

best_score = -1.0
best_acc = -1.0
best_params = (0.5, 2.2, 0.5)
train_prior = y_cls.mean()

for scale in [1.8, 2.0, 2.2, 2.5, 3.0]:
    reg_prob_oof = expit((oof_reg_blend - 60.0) / scale)
    for alpha in np.linspace(0.2, 0.85, 14):
        combined_oof = alpha * reg_prob_oof + (1.0 - alpha) * oof_cls_blend
        for th in np.linspace(0.46, 0.54, 81):
            preds_bin = (combined_oof >= th).astype(int)
            acc = accuracy_score(y_cls, preds_bin)
            pos_rate = preds_bin.mean()
            prior_penalty = 0.05 * max(0.0, abs(pos_rate - train_prior) - 0.025)
            score = acc - prior_penalty
            if score > best_score:
                best_score = score
                best_acc = acc
                best_params = (alpha, scale, th)

best_alpha, best_scale, best_th = best_params
final_oof_prob = best_alpha * expit((oof_reg_blend - 60.0) / best_scale) + (1.0 - best_alpha) * oof_cls_blend
final_oof_pred = (final_oof_prob >= best_th).astype(int)

final_test_prob = best_alpha * expit((test_reg_blend - 60.0) / best_scale) + (1.0 - best_alpha) * test_cls_blend
final_test_pred = (final_test_prob >= best_th)

oof_auc = roc_auc_score(y_cls, final_oof_prob)
oof_f1 = f1_score(y_cls, final_oof_pred)
cm = confusion_matrix(y_cls, final_oof_pred)

log_print("\n")
log_print("FINAL V2 META-ENSEMBLE OOF RESULTS")
log_print("")
log_print(f"  -> Optimal Blend Params : alpha(Reg)={best_alpha:.2f}, scale={best_scale:.1f}, threshold={best_th:.4f}")
log_print(f"  -> Final OOF Accuracy   : {best_acc*100:.4f}%")
log_print(f"  -> Final OOF F1-Score   : {oof_f1:.4f}")
log_print(f"  -> Final OOF ROC-AUC    : {oof_auc:.5f}")
log_print(f"  -> OOF Viable Rate      : {final_oof_pred.mean()*100:.2f}% (Train True Rate: {train_prior*100:.2f}%)")
log_print(f"  -> Confusion Matrix     :\n{cm}")
log_print("")


FINAL V2 META-ENSEMBLE OOF RESULTS

  -> Optimal Blend Params : alpha(Reg)=0.80, scale=1.8, threshold=0.5240
  -> Final OOF Accuracy   : 88.8438%
  -> Final OOF F1-Score   : 0.8643
  -> Final OOF ROC-AUC    : 0.94748
  -> OOF Viable Rate      : 39.91% (Train True Rate: 42.31%)
  -> Confusion Matrix     :
[[3412  280]
 [ 434 2274]]




In [1]:
# Prediction file generation

out_dir = '/content/submission_bonus' if os.path.exists('/content') else './submission_bonus'
os.makedirs(out_dir, exist_ok=True)

sub_cols = list(sub_raw.columns)
id_col_name = sub_cols[0]
target_col_name = sub_cols[1]

pred_df = pd.DataFrame({
    id_col_name: test_raw['planetId'].values,
    target_col_name: final_test_pred.astype(bool)
})

pred_csv_path = os.path.join(out_dir, 'The_gentlemen_predicted.csv')
pred_df.to_csv(pred_csv_path, index=False)

log_print(f"\n[Step 6] Saved predictions to: {pred_csv_path}")
log_print(f"  -> Columns: {list(pred_df.columns)} | Shape: {pred_df.shape}")
log_print(f"  -> Predicted Viable (True): {pred_df[target_col_name].sum()} ({pred_df[target_col_name].mean()*100:.2f}%)")
log_print("  -> First 10 rows:")
log_print(pred_df.head(10).to_string(index=False))


[Step 6] Saved predictions to: /content/submission_bonus/The_gentlemen_predicted.csv
  -> Columns: ['planetId', 'viable_planet'] | Shape: (1600, 2)
  -> Predicted Viable (True): 653 (40.81%)
  -> First 10 rows:
 planetId  viable_planet
        9          False
       10          False
       11          False
       12           True
       17          False
       25           True
       29          False
       31           True
       43           True
       47           True
